In [1]:
import scanpy as sc 
import numpy as np
import pandas as pd

In [2]:
import pathlib
import zipfile

In [3]:
pwd

'/home/pandeyps/Prefix/scM/quality control/prostate'

In [4]:
# %pip install scrublet

In [5]:
import scrublet as scr 
import os

In [6]:
# 4 TOTAL STUDIES

In [7]:
study_files = {
    "Chen2021": {
        "matrix": "Data_Chen2021_Prostate/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Chen2021_Prostate/Genes.txt",
        "cells":  "Data_Chen2021_Prostate/Cells.csv",
        "samples": "Data_Chen2021_Prostate/Samples.csv",
        "units": "UMI"
    },
    "Dong2020": {
        "matrix": "Data_Dong2020_Prostate/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Dong2020_Prostate/Genes.txt",
        "cells":  "Data_Dong2020_Prostate/Cells.csv",
        "samples": "Data_Dong2020_Prostate/Samples.csv",
        "units": "UMI"
    },
    "He2021": {
        "matrix": "Data_He2021_Prostate/Exp_data_TPM.mtx",
        "genes":  "Data_He2021_Prostate/Genes.txt",
        "cells":  "Data_He2021_Prostate/Cells.csv",
        "samples": "Data_He2021_Prostate/Samples.csv",
        "units": "TPM"
    },
    "Song2022": {
        "matrix": "Data_Song2022_Prostate/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Song2022_Prostate/Genes.txt",
        "cells":  "Data_Song2022_Prostate/Cells.csv",
        "samples": "Data_Song2022_Prostate/Samples.csv",
        "units": "UMI"
    }
}

In [12]:
# STUDY 1

In [13]:
study_name = 'Chen2021'
paths = study_files[study_name]

In [14]:
pwd

'/home/pandeyps/Prefix/scM/quality control/prostate'

In [15]:
# creating matrix (cells x gene)
adata = sc.read_mtx(paths['matrix']).T

In [16]:
# gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes

In [17]:
# cell metadata
cells = pd.read_csv(paths["cells"], index_col=0)
adata.obs = adata.obs.join(cells, how="left")
adata.obs_names = cells.index   #barcodes should match

In [18]:
# adding study column
adata.obs["study"] = study_name

In [19]:
f'{adata.n_obs:,} cells, {adata.n_vars:,} genes'

'36,424 cells, 25,044 genes'

In [20]:
# GENE COUNT FILTER (200-6000)
sc.pp.calculate_qc_metrics(adata, inplace=True)

min_genes = 200
max_genes = 6000
keep = (adata.obs["n_genes_by_counts"] >= min_genes) & \
       (adata.obs["n_genes_by_counts"] <= max_genes)
adata = adata[keep, :].copy()

In [21]:
f" {adata.n_obs:,} cells"

' 36,424 cells'

In [22]:
# DOUBLET REMOVAL USING SCRUBLET
# scrublet needs dense array, our matrix is sparse, so converting it to an array
counts = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X

In [23]:
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()

Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.58
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 9.3%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.9%
Elapsed time: 53.7 seconds


In [24]:
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets

In [25]:
adata = adata[~adata.obs["predicted_doublet"], :].copy()

In [26]:
f"   After doublet removal- {adata.n_obs:,} cells"

'   After doublet removal- 36,393 cells'

In [27]:
# Dead cell removal
adata.var['mt'] = adata.var_names.str.startswith('MT-') #get mitochondrial genes
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
max_pct_mt = 15
keep_mt = adata.obs['pct_counts_mt'] <= max_pct_mt
adata = adata[keep_mt, :].copy()

In [28]:
f"after MT filter: {adata.n_obs:,} cells"

'after MT filter: 36,393 cells'

In [29]:
# Convert columns to string (conflict for h5ad)
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

In [30]:
adata.write_h5ad('prostate_study1_qc.h5ad')

In [31]:
# STUDY 2

In [32]:
study_name = 'Dong2020'
paths = study_files[study_name]

# matrix,gene and cell
adata = sc.read_mtx(paths['matrix']).T
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes

cells = pd.read_csv(paths["cells"], index_col=0)
adata.obs = adata.obs.join(cells, how="left")
adata.obs_names = cells.index   # barcodes should match

adata.obs["study"] = study_name
print(f"{adata.n_obs:,} cells, {adata.n_vars:,} genes")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
min_genes = 200
max_genes = 6000
keep = (adata.obs["n_genes_by_counts"] >= min_genes) & \
       (adata.obs["n_genes_by_counts"] <= max_genes)
adata = adata[keep, :].copy()
print(f"after gene filter {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()

adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"after doublet removal: {adata.n_obs:,} cells")

# Dead cell removal
adata.var['mt'] = adata.var_names.str.startswith('MT-') #get mitochondrial genes
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
max_pct_mt = 15
keep_mt = adata.obs['pct_counts_mt'] <= max_pct_mt
adata = adata[keep_mt, :].copy()
print(f"after MT filter: {adata.n_obs:,} cells")

# conversion to string
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

adata.write_h5ad('prostate_study2_qc.h5ad')

21,292 cells, 15,709 genes
after gene filter 21,292 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.60
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 18.8%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.1%
Elapsed time: 19.8 seconds
after doublet removal: 21,286 cells
after MT filter: 20,820 cells


In [33]:
# STUDY 3

In [34]:
study_name = 'He2021'
paths = study_files[study_name]

# matrix,gene and cell
adata = sc.read_mtx(paths['matrix']).T
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes

cells = pd.read_csv(paths["cells"], index_col=0)
adata.obs = adata.obs.join(cells, how="left")
adata.obs_names = cells.index   # barcodes should match
adata.obs_names = adata.obs_names.astype(str) #this study uses integer as cell index

adata.obs["study"] = study_name
print(f"{adata.n_obs:,} cells, {adata.n_vars:,} genes")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
min_genes = 200
max_genes = 6000
keep = (adata.obs["n_genes_by_counts"] >= min_genes) & \
       (adata.obs["n_genes_by_counts"] <= max_genes)
adata = adata[keep, :].copy()
print(f"after gene filter {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()

adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"after doublet removal: {adata.n_obs:,} cells")

# Dead cell removal
adata.var['mt'] = adata.var_names.str.startswith('MT-') #get mitochondrial genes
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
max_pct_mt = 15
keep_mt = adata.obs['pct_counts_mt'] <= max_pct_mt
adata = adata[keep_mt, :].copy()
print(f"after MT filter: {adata.n_obs:,} cells")

# conversion to string
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

adata.write_h5ad('prostate_study3_qc.h5ad')

/home/pandeyps/.pyenv/versions/3.11.9/lib/python3.11/site-packages/anndata/_core/anndata.py:879: UserWarning: 
AnnData expects .obs.index to contain strings, but got values like:
    [0, 1, 2, 3, 4]

    Inferred to be: integer

  names = self._prep_dim_index(names, "obs")


2,170 cells, 45,895 genes
after gene filter 2,151 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.42
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 43.0%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.0%
Elapsed time: 3.0 seconds
after doublet removal: 2,151 cells
after MT filter: 2,151 cells


In [35]:
study_name = 'Song2022'
paths = study_files[study_name]

# matrix,gene and cell
adata = sc.read_mtx(paths['matrix']).T
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes

cells = pd.read_csv(paths["cells"], index_col=0)
adata.obs = adata.obs.join(cells, how="left")
adata.obs_names = cells.index   # barcodes should match
adata.obs_names = adata.obs_names.astype(str) #this study uses integer as cell index

adata.obs["study"] = study_name
print(f"{adata.n_obs:,} cells, {adata.n_vars:,} genes")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
min_genes = 200
max_genes = 6000
keep = (adata.obs["n_genes_by_counts"] >= min_genes) & \
       (adata.obs["n_genes_by_counts"] <= max_genes)
adata = adata[keep, :].copy()
print(f"after gene filter {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()

adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"after doublet removal: {adata.n_obs:,} cells")

# Dead cell removal
adata.var['mt'] = adata.var_names.str.startswith('MT-') #get mitochondrial genes
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
max_pct_mt = 15
keep_mt = adata.obs['pct_counts_mt'] <= max_pct_mt
adata = adata[keep_mt, :].copy()
print(f"after MT filter: {adata.n_obs:,} cells")

# conversion to string
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

adata.write_h5ad('prostate_study4_qc.h5ad')

21,743 cells, 21,877 genes
after gene filter 21,540 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.72
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 0.5%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 3.4%
Elapsed time: 19.1 seconds
after doublet removal: 21,536 cells
after MT filter: 20,355 cells


In [8]:
# NORMALIZE ALL STUDIES

In [9]:
import glob

In [10]:
files = {
    "prostate_study1_qc.h5ad": "Chen2021",
    "prostate_study2_qc.h5ad": "Dong2020",
    "prostate_study3_qc.h5ad": "He2021",
    "prostate_study4_qc.h5ad": "Song2022"
}

for fname, study_name in files.items():
    unit = study_files[study_name]["units"]
    adata = sc.read_h5ad(fname)

    if unit == "TPM":
        sc.pp.log1p(adata)
    else:
        sc.pp.normalize_total(adata, target_sum=1e6)
        sc.pp.log1p(adata)

    out_name = f"{study_name}_normalized.h5ad"
    adata.write_h5ad(out_name)
    print(f"{study_name}: normalised ({unit}) – max value {adata.X.max():.2f}")

Chen2021: normalised (UMI) – max value 13.23
Dong2020: normalised (UMI) – max value 13.54
He2021: normalised (TPM) – max value 12.98
Song2022: normalised (UMI) – max value 12.87
